# Download La Manno Human Brain Dataset

**Quick and easy way to get RNA velocity data with spliced/unspliced counts**

Dataset: Human embryonic forebrain (week 10)  
Paper: La Manno et al. Nature 2018  
Cells: ~1,720 glutamatergic neurons

## Important Setup Instructions

**FIRST TIME ONLY**: Run the first cell, then **RESTART JULIA** and re-run the notebook.

This configures PyCall to use a Julia-managed Python environment.

## Step 1: Configure PyCall (First Time Only)

**Run this cell ONCE, then RESTART Julia**

This sets up PyCall to use Julia's Conda Python (easier than system Python).

In [16]:
# Setup - Configure PyCall to use Conda.jl
using Pkg

# Install Conda and PyCall if needed
for pkg in ["Conda", "PyCall"]
    try
        eval(:(using $(Symbol(pkg))))
    catch
        Pkg.add(pkg)
    end
end

# Configure PyCall to use Julia's Conda Python
ENV["PYTHON"] = ""
Pkg.build("PyCall")

println("✓ PyCall configured to use Conda")
println("⚠️  You must RESTART Julia now for this to take effect!")
println("    Then re-run this notebook from the beginning.")

    Building Conda ─→ `~/.julia/scratchspaces/44cfe95a-1eb2-52ea-b672-e2afdf69b78f/8f06b0cfa4c514c7b9546756dbae91fcfbc92dc9/build.log`
    Building PyCall → `~/.julia/scratchspaces/44cfe95a-1eb2-52ea-b672-e2afdf69b78f/9816a3826b0ebf49ab4926e2b18842ad8b5c8f04/build.log`


✓ PyCall configured to use Conda
⚠️  You must RESTART Julia now for this to take effect!
    Then re-run this notebook from the beginning.


## Step 2: Download a Neuro Velocity Dataset

This tries **forebrain** first. If the upstream forebrain URL is broken, it falls back to **dentategyrus** (also neural, with spliced/unspliced layers).


In [23]:
using PyCall, Conda

println("Installing/importing scvelo...")
try
    scv = pyimport("scvelo")
catch
    Conda.add("scvelo", channel="conda-forge")
    scv = pyimport("scvelo")
end

# Keep track of what we end up loading
dataset_name = ""

println("\nTrying FOREBRAIN (La Manno 2018)...")
try
    adata = scv.datasets.forebrain()
    dataset_name = "forebrain"
    println("✓ Loaded forebrain")
catch e
    println("Forebrain unavailable right now.")
    println("Reason: ", e)
    println("\nFalling back to DENTATEGYRUS (neuro dataset)...")
    adata = scv.datasets.dentategyrus()
    dataset_name = "dentategyrus"
    println("✓ Loaded dentategyrus")
end

println("\nDataset: ", dataset_name)
println("Cells: ", adata.n_obs)
println("Genes: ", adata.n_vars)

# Correct Python-side layer introspection
layer_names = String.(collect(adata.layers.keys()))
println("Layers: ", layer_names)
println("Has spliced: ", "spliced" in layer_names)
println("Has unspliced: ", "unspliced" in layer_names)


Installing/importing scvelo...

Trying FOREBRAIN (La Manno 2018)...
Forebrain unavailable right now.
Reason: PyError ($(Expr(:escape, :(ccall(#= /home/antorios/.julia/packages/PyCall/1gn3u/src/pyfncall.jl:43 =# @pysym(:PyObject_Call), PyPtr, (PyPtr, PyPtr, PyPtr), o, pyargsptr, kw))))) <class 'OSError'>
OSError('Unable to synchronously open file (file signature not found)')
  File "/home/antorios/.julia/conda/3/x86_64/lib/python3.13/site-packages/scvelo/datasets/_datasets.py", line 163, in forebrain
    adata = read(file_path, backup_url=url, cleanup=True, sparse=True, cache=True)
  File "/home/antorios/.julia/conda/3/x86_64/lib/python3.13/site-packages/legacy_api_wrap/__init__.py", line 88, in fn_compatible
    return fn(*args_all, **kw)
  File "/home/antorios/.julia/conda/3/x86_64/lib/python3.13/site-packages/scanpy/readwrite.py", line 150, in read
    return _read(
        filename,
    ...<8 lines>...
        **kwargs,
    )
  File "/home/antorios/.julia/conda/3/x86_64/lib/python3.

## Step 2b: Sanity Check (optional)

Run this only if you want to confirm which dataset is currently loaded and whether velocity layers exist.


In [24]:
if @isdefined adata
    println("Current dataset loaded")
    println("Cells: ", adata.n_obs)
    println("Genes: ", adata.n_vars)
    if @isdefined dataset_name
        println("Dataset name: ", dataset_name)
    end

    layer_names = String.(collect(adata.layers.keys()))
    println("Layers: ", layer_names)
    println("Has spliced: ", "spliced" in layer_names)
    println("Has unspliced: ", "unspliced" in layer_names)
else
    println("No dataset loaded yet. Run Step 2 first.")
end


Current dataset loaded
Cells: 2930
Genes: 13913
Dataset name: dentategyrus
Layers: ["ambiguous", "spliced", "unspliced"]
Has spliced: true
Has unspliced: true


## Step 3: Save to `h5ad`


In [25]:
DATA_DIR = "lamanno_data"
mkpath(DATA_DIR)

if !isdefined(Main, :adata)
    error("No dataset found. Run Step 2 first.")
end

name = (isdefined(Main, :dataset_name) && !isempty(dataset_name)) ? dataset_name : "neuro_dataset"
output_file = joinpath(DATA_DIR, "$(name)_velocity.h5ad")
adata.write(output_file)

println("✓ Saved: ", output_file)
println("Size: ", round(filesize(output_file)/1e6, digits=1), " MB")


✓ Saved: lamanno_data/dentategyrus_velocity.h5ad
Size: 86.2 MB


## Step 4: Verify Saved File via Python (no Muon)

This avoids local `Muon`/`HDF5_jll` binary issues and verifies the saved `h5ad` directly with `anndata` through PyCall.


In [26]:
using PyCall

name = (isdefined(Main, :dataset_name) && !isempty(dataset_name)) ? dataset_name : "neuro_dataset"
h5ad_file = joinpath("lamanno_data", "$(name)_velocity.h5ad")
println("Loading via anndata: ", h5ad_file)

if !isfile(h5ad_file)
    error("Saved file not found. Run Step 3 first.")
end

anndata = pyimport("anndata")
adata_loaded = anndata.read_h5ad(h5ad_file)

println("Cells: ", Int(adata_loaded.n_obs))
println("Genes: ", Int(adata_loaded.n_vars))
layer_names_loaded = String.(collect(adata_loaded.layers.keys()))
println("Layers: ", layer_names_loaded)


Loading via anndata: lamanno_data/dentategyrus_velocity.h5ad
Cells: 2930
Genes: 13913
Layers: ["ambiguous", "spliced", "unspliced"]


## Step 5: Check Spliced/Unspliced Layers (from saved file)


In [33]:
# Robust PyCall-only layer access (no py"..." assignment)
using PyCall

layers = adata_loaded[:layers]

# Get layer names safely
layer_names_py = pycall(layers[:keys], PyObject)
layer_names = String.(collect(layer_names_py))

has_spliced = "spliced" in layer_names
has_unspliced = "unspliced" in layer_names

println("Has spliced: ", has_spliced)
println("Has unspliced: ", has_unspliced)
println("Layer names: ", layer_names)

if has_spliced && has_unspliced
    spliced = pycall(layers[:get], PyObject, "spliced")
    unspliced = pycall(layers[:get], PyObject, "unspliced")

    ssum = pycall(spliced[:sum], Float64)
    usum = pycall(unspliced[:sum], Float64)
    frac_u = usum / (ssum + usum) * 100

    println("Spliced total: ", round(ssum, digits=2))
    println("Unspliced total: ", round(usum, digits=2))
    println("Unspliced fraction: ", round(frac_u, digits=2), "%")
    println("\n✅ Ready for RNA velocity.")
else
    println("❌ Missing velocity layers.")
end


Has spliced: true
Has unspliced: true
Layer names: ["ambiguous", "spliced", "unspliced"]
Spliced total: 8.437248e6
Unspliced total: 974602.0
Unspliced fraction: 10.36%

✅ Ready for RNA velocity.


## Notes

- If forebrain is temporarily unavailable upstream, this notebook automatically uses **dentategyrus** (still neuro).
- You can now plug `adata_julia` into your existing velocity pipeline.
